In [ ]:
import geopandas as gpd
import pandas as pd

fp = r"C:\Users\gabri\OneDrive\Ambiente de Trabalho\2ºsemestre\Seminário\Trabalho Seminário\Trabalho_Seminário\MstCSCS_Sem_2526.gpkg"

# 1. Carregar a camada do CP7 e converter o ID para string
df_base = gpd.read_file(fp, layer='postal_code_buildings_assigned')
df_base['polygon_id'] = df_base['polygon_id'].astype(str) # Conversão aqui

# 2. Carregar a camada da Altura e converter o ID para string
df_altura = gpd.read_file(fp, layer='ed12_polygons_all_with_heights_clean')
df_altura['polygon_id'] = df_altura['polygon_id'].astype(str) # Conversão aqui

# 3. Agora o merge funcionará corretamente
df_final = df_base[['polygon_id', 'cp7', 'building_area_m2', 'geometry']].merge(
    df_altura[['polygon_id', 'altura_edif_m']], 
    on='polygon_id', 
    how='left'
)

print(df_final.head())

  polygon_id       cp7  building_area_m2  \
0     100024  3800-200         99.080842   
1     100032  3810-086        373.299217   
2     100039  3800-176         20.819647   
3      10004  3800-302        155.768200   
4      10005  3800-041        176.255125   

                                            geometry  altura_edif_m  
0  MULTIPOLYGON (((-44096.203 108224.566, -44089....      14.000000  
1  MULTIPOLYGON (((-44021.294 108095.965, -44029....      13.530001  
2  MULTIPOLYGON (((-43378.759 108515.018, -43381....       4.370001  
3  MULTIPOLYGON (((-40290.943 110059.539, -40290....       8.259998  
4  MULTIPOLYGON (((-41381.122 110997.515, -41377....       5.820000  


In [ ]:
import geopandas as gpd
import pandas as pd

# 1. Definição do ficheiro de entrada

fp = r"C:\Users\gabri\OneDrive\Ambiente de Trabalho\2ºsemestre\Seminário\Trabalho Seminário\Trabalho_Seminário\MstCSCS_Sem_2526.gpkg"

print("A carregar camadas...")

# 2. Carregar Camada de Códigos Postais e Áreas
# Esta camada contém a ligação geográfica aos CP7
df_base = gpd.read_file(fp, layer='postal_code_buildings_assigned')
# Selecionamos apenas as colunas que interessam para o cruzamento
df_base = df_base[['polygon_id', 'cp7', 'building_area_m2', 'geometry']]

# 3. Carregar Camada de Alturas
# Esta camada contém a informação vertical dos edifícios
df_altura = gpd.read_file(fp, layer='ed12_polygons_all_with_heights_clean')
# Selecionamos apenas o ID e a Altura
df_altura = df_altura[['polygon_id', 'altura_edif_m', 'Layer']]

# 4. TRATAMENTO DE ERROS: Harmonização de Tipos
# Convertemos os IDs para string em ambas as tabelas para evitar o erro de 'ValueError'
df_base['polygon_id'] = df_base['polygon_id'].astype(str)
df_altura['polygon_id'] = df_altura['polygon_id'].astype(str)

print("A realizar a junção das colunas...")

# 5. JUNÇÃO (Merge)
# Criamos um único GeoDataFrame com toda a informação técnica do edifício
edificios_db = df_base.merge(df_altura, on='polygon_id', how='left')

# Opcional: Renomear colunas para nomes mais amigáveis no código
edificios_db = edificios_db.rename(columns={
    'polygon_id': 'id_edificio',
    'building_area_m2': 'area_m2',
    'altura_edif_m': 'altura_m'
})

# 6. Verificação dos resultados
print("\nBase de dados de edifícios preparada:")
print(edificios_db.head())
print(f"\nTotal de edifícios processados: {len(edificios_db)}")



A carregar camadas...
A realizar a junção das colunas...

Base de dados de edifícios preparada:
  id_edificio       cp7     area_m2  \
0      100024  3800-200   99.080842   
1      100032  3810-086  373.299217   
2      100039  3800-176   20.819647   
3       10004  3800-302  155.768200   
4       10005  3800-041  176.255125   

                                            geometry   altura_m  \
0  MULTIPOLYGON (((-44096.203 108224.566, -44089....  14.000000   
1  MULTIPOLYGON (((-44021.294 108095.965, -44029....  13.530001   
2  MULTIPOLYGON (((-43378.759 108515.018, -43381....   4.370001   
3  MULTIPOLYGON (((-40290.943 110059.539, -40290....   8.259998   
4  MULTIPOLYGON (((-41381.122 110997.515, -41377....   5.820000   

                  Layer  
0     ED07_EDIF_NOTAVEL  
1     ED07_EDIF_NOTAVEL  
2     ED07_EDIF_NOTAVEL  
3  ED08_EDIF_PERMANENTE  
4  ED08_EDIF_PERMANENTE  

Total de edifícios processados: 17559


In [ ]:
import geopandas as gpd
import pandas as pd

# 1. Definição do caminho do ficheiro

fp = r"C:\Users\gabri\OneDrive\Ambiente de Trabalho\2ºsemestre\Seminário\Trabalho Seminário\Trabalho_Seminário\MstCSCS_Sem_2526.gpkg"

# 2. Carregar a camada com 17 mil edifícios (a que tem o CP7)
# Vamos selecionar apenas as colunas polygon_id e cp7 para o cruzamento
camada_cp7 = gpd.read_file(fp, layer='postal_code_buildings_assigned')
df_cp7 = camada_cp7[['polygon_id', 'cp7']]

# 3. Carregar a camada com 88 mil edifícios (a que tem alturas e áreas)
camada_completa = gpd.read_file(fp, layer='ed12_polygons_all_with_heights_clean')
# Selecionamos o ID, a altura e a geometria (e a área, se existir nesta camada)
df_88k = camada_completa[['polygon_id', 'altura_edif_m', 'area_m2', 'geometry','Layer']]

# 4. CORREÇÃO DO ERRO: Converter polygon_id para string em ambos os DataFrames
# Isto evita o erro de "merge on str and int64" que encontraste anteriormente
df_cp7['polygon_id'] = df_cp7['polygon_id'].astype(str)
df_88k['polygon_id'] = df_88k['polygon_id'].astype(str)

# 5. CRUZAMENTO (Merge)
# Usamos 'left' para manter todos os 88 mil edifícios. 
# O CP7 será preenchido onde houver correspondência com os 17 mil.
edificios_final = df_88k.merge(df_cp7, on='polygon_id', how='left')

# 6. VERIFICAÇÃO
print(f"Total de edifícios no ficheiro final: {len(edificios_final)}")
print(f"Edifícios com CP7 atribuído: {edificios_final['cp7'].notna().sum()}")

# Visualizar as primeiras linhas
edificios_final.head()

Total de edifícios no ficheiro final: 90189
Edifícios com CP7 atribuído: 17559


C:\Users\gabri\AppData\Local\Temp\ipykernel_17752\2949270910.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cp7['polygon_id'] = df_cp7['polygon_id'].astype(str)
c:\Users\gabri\miniforge3\envs\envEOAD_statsPythonR\Lib\site-packages\geopandas\geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,polygon_id,altura_edif_m,area_m2,geometry,Layer,cp7
0,0,3.30,31.74,"POLYGON Z ((-49484.997 115012.735 0, -49483.04...",ED08_EDIF_PERMANENTE,NaN
1,1,3.61,44.85,"POLYGON Z ((-49498.795 115017.57 0, -49504.374...",ED08_EDIF_PERMANENTE,NaN
2,4,3.10,149.50,"POLYGON Z ((-49373.264 115004.713 0, -49374.9 ...",ED08_EDIF_PERMANENTE,NaN
3,5,4.33,153.76,"POLYGON Z ((-49348.917 115014.366 0, -49352.77...",ED08_EDIF_PERMANENTE,NaN
4,6,3.25,11.45,"POLYGON Z ((-49435.132 115040.856 0, -49437.43...",ED08_EDIF_PERMANENTE,NaN


In [ ]:
import geopandas as gpd

# 1. Ler o ficheiro GeoJSON (o que descarregaste do Overpass Turbo)
# Substitui 'export.geojson' pelo nome real do teu ficheiro
limites_postais = gpd.read_file(r"C:\Users\gabri\OneDrive\Ambiente de Trabalho\2ºsemestre\Seminário\Trabalho Seminário\Trabalho_Seminário\export.geojson")

# 2. Carregar a tua base de edifícios (os 90.189 totais do teu merge anterior)
# Vamos usar o GeoPackage original como fonte da geometria
edificios = gpd.read_file(r"C:\Users\gabri\OneDrive\Ambiente de Trabalho\2ºsemestre\Seminário\Trabalho Seminário\Trabalho_Seminário\MstCSCS_Sem_2526.gpkg", layer="ed12_polygons_all_with_heights_clean")

# 3. HARMONIZAÇÃO DE COORDENADAS (Passo Crítico)
# O GeoJSON do Overpass costuma vir em WGS84 (EPSG:4326)
# O teu GPKG está em ETRS89 / Portugal TM06 (EPSG:3763)
limites_postais = limites_postais.to_crs(edificios.crs)

print("A realizar a junção espacial...")

# 4. JUNÇÃO ESPACIAL (Spatial Join)
# Atribuímos o código postal ao edifício que interseta o polígono do Overpass
# Nota: Verifica no teu GeoJSON se a coluna se chama 'postal_code' ou 'addr:postcode'
coluna_cp = 'postal_code' if 'postal_code' in limites_postais.columns else 'addr:postcode'

edificios_final = gpd.sjoin(
    edificios, 
    limites_postais[[coluna_cp, 'geometry']], 
    how="left", 
    predicate="intersects"
)

# 5. RESULTADOS
total_com_cp = edificios_final[coluna_cp].notna().sum()
print(f"Total de edifícios no sistema: {len(edificios_final)}")
print(f"Edifícios com CP atribuído via Spatial Join: {total_com_cp}")

# Guardar o resultado para não teres de repetir o processo
# edificios_final.to_file("Base_Edificios_Aveiro_Final.gpkg", driver="GPKG")

A realizar a junção espacial...
Total de edifícios no sistema: 88784
Edifícios com CP atribuído via Spatial Join: 423


In [ ]:
import geopandas as gpd
from scipy.spatial import cKDTree

# 1. Carregar as duas camadas do teu GeoPackage
edificios_total = gpd.read_file(r"C:\Users\gabri\OneDrive\Ambiente de Trabalho\2ºsemestre\Seminário\Trabalho Seminário\Trabalho_Seminário\MstCSCS_Sem_2526.gpkg", layer="ed12_polygons_all_with_heights_clean")
edificios_com_cp = gpd.read_file(r"C:\Users\gabri\OneDrive\Ambiente de Trabalho\2ºsemestre\Seminário\Trabalho Seminário\Trabalho_Seminário\MstCSCS_Sem_2526.gpkg", layer="postal_code_buildings_assigned")

# 2. Preparar os pontos para o cálculo de distância
# Usamos o centroide (ponto central) de cada edifício para calcular a proximidade
conhecidos = edificios_com_cp.copy()
conhecidos['centroid'] = conhecidos.geometry.centroid
desconhecidos = edificios_total.copy()
desconhecidos['centroid'] = desconhecidos.geometry.centroid

# 3. Criar a "Árvore de Procura" (cKDTree) para rapidez matemática
# Extraímos as coordenadas (x, y) dos edifícios que já têm CP7
coords_conhecidas = list(zip(conhecidos.centroid.x, conhecidos.centroid.y))
arvore_espacial = cKDTree(coords_conhecidas)

# 4. Procurar o vizinho mais próximo para todos os 88.784 edifícios
coords_alvo = list(zip(desconhecidos.centroid.x, desconhecidos.centroid.y))
distancias, indices = arvore_espacial.query(coords_alvo)

# 5. Atribuir o CP7 do vizinho mais próximo
desconhecidos['cp7_estimado'] = conhecidos.iloc[indices]['cp7'].values

# 6. Limpeza Final
# Mantemos as colunas originais e a geometria
edificios_aveiro_completo = desconhecidos[['polygon_id', 'altura_edif_m', 'area_m2', 'cp7_estimado', 'geometry','Layer']]

print(f"Total de edifícios processados: {len(edificios_aveiro_completo)}")
print(f"Exemplo de atribuição:\n{edificios_aveiro_completo[['polygon_id', 'cp7_estimado']].head()}")

Total de edifícios processados: 88649
Exemplo de atribuição:
   polygon_id cp7_estimado
0           0     3800-903
1           1     3800-903
2           4     3800-903
3           5     3800-903
4           6     3800-903


In [ ]:
# Exemplo do que faremos a seguir:
# 1. Carregar os consumos
df_consumos = pd.read_csv(r"C:\Users\gabri\OneDrive\Ambiente de Trabalho\2ºsemestre\Seminário\Trabalho Seminário\Trabalho_Seminário\consumo_aveiro_pre_processed.csv")

# 2. Cruzar com a tua base de 88.649 edifícios
# Usamos 'left' para garantir que mantemos a geometria de todos os edifícios
df_consumo_edificio = edificios_aveiro_completo.merge(
    df_consumos, 
    left_on='cp7_estimado', 
    right_on='Código Postal', 
    how='left'
)

In [ ]:
# Agrupar consumos por Código Postal somando os valores
df_consumos_agrupado = df_consumos.groupby('Código Postal')['Energia ativa (kWh)'].sum().reset_index()

# Agora o merge terá uma relação de 1 para 1
df_consumo_edificio = edificios_aveiro_completo.merge(
    df_consumos_agrupado, 
    left_on='cp7_estimado', 
    right_on='Código Postal', 
    how='left'
)

In [ ]:
df_consumos_agrupado.head(5)

In [ ]:
df_consumo_edificio.to_csv("Base_Edificios_Aveiro_Com_Consumos.csv", index=False)